# ARC-v0.7 — HotpotQA Shard-Based Fidelity Index Rebuild

Rebuilds **IVF-PQ32, IVF-PQ64, and IVF-SQ8** from the new verified HotpotQA embedding shards.

It never reads the legacy zero-filled corpus memmap.

Pipeline:

1. validate shard manifest;
2. reload DEV queries/qrels;
3. positive-vs-random representation gate;
4. deterministic 200k training sample;
5. build PQ32/PQ64/SQ8 from shards;
6. IVF population audit;
7. 100-query smoke test;
8. full 5,447-query DEV baseline;
9. save hashes and aggregate evidence.

Stop before H1–H4. Continue only if all baseline validity gates pass.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [ ]:
%pip install -q faiss-cpu==1.12.0 psutil pyarrow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 87.5 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, gc, sqlite3, time

import numpy as np
import pandas as pd
import faiss
import psutil

SEED = 20260816
DIM = 384
N_DOCS = 5_233_329
NLIST = 4096
NPROBE = 64
NBITS = 8
TRAIN_DOCS = 200_000
TOP_K = 10
SEARCH_K = 100

ROOT = Path("/content/drive/MyDrive/hc-rars-external-confirmation-hotpotqa-5m-v1")
SHARD_ROOT = ROOT / "stage1/corpus-embedding-shards-v3"
SHARD_MANIFEST = SHARD_ROOT / "manifest.json"
QUERY_EMB = ROOT / "stage1/query_embeddings.float32.npy"
QUERY_IDS = ROOT / "stage1/query_ids.utf8.txt"
SPLIT_MANIFEST = ROOT / "stage1/official_split_manifest.json"
CORPUS_DB = ROOT / "stage1/corpus_ids.sqlite"
DEV_QRELS = ROOT / "source/hotpotqa/qrels/dev.tsv"

CACHE_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-index-cache/hotpotqa-rebuilt-v3")
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

OUT_ROOT = Path("/content/drive/MyDrive/rag-pq-checkpoints/arc-v0/hotpotqa-fidelity-index-rebuild-v07")
OUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
OUT = OUT_ROOT / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)

PQ32_PATH = CACHE_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m32-nbits8-seed20260816.faiss"
PQ64_PATH = CACHE_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfpq-nlist4096-m64-nbits8-seed20260816.faiss"
SQ8_PATH  = CACHE_ROOT / "hotpotqa-v3-5233329d-bge-small-ivfsq8-nlist4096-seed20260816.faiss"

for p in [SHARD_MANIFEST, QUERY_EMB, QUERY_IDS, SPLIT_MANIFEST, CORPUS_DB, DEV_QRELS]:
    if not p.is_file():
        raise FileNotFoundError(p)

print("FAISS:", faiss.__version__)
print("Output:", OUT)


FAISS: 1.12.0
Output: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/hotpotqa-fidelity-index-rebuild-v07/20260816-101236


## 1. Validate rebuilt corpus shards


In [ ]:
def sha256_file(path, chunk_size=64 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

with open(SHARD_MANIFEST, "r", encoding="utf-8") as f:
    shard_manifest = json.load(f)

assert shard_manifest.get("status") == "CORPUS_EMBEDDING_SHARDS_COMPLETE"

shards = sorted(shard_manifest["shards"], key=lambda x: int(x["shard_id"]))

expected_start = 0
total_rows = 0
for s in shards:
    start = int(s["start_row"])
    end = int(s["end_row"])
    rows = int(s["rows"])
    path = SHARD_ROOT / s["file"]

    assert start == expected_start
    assert end - start == rows
    assert path.is_file()
    assert path.stat().st_size == int(s["bytes"])

    total_rows += rows
    expected_start = end

assert total_rows == N_DOCS
assert expected_start == N_DOCS
assert int(shard_manifest["dimension"]) == DIM

print("shards:", len(shards))
print("total rows:", total_rows)
print("SHARD MANIFEST — PASS")


shards: 262
total rows: 5233329
SHARD MANIFEST — PASS


In [ ]:
def load_rows_from_shards(rows):
    rows = np.asarray(rows, dtype=np.int64)
    out = np.empty((len(rows), DIM), dtype=np.float32)
    filled = np.zeros(len(rows), dtype=bool)

    for s in shards:
        start = int(s["start_row"])
        end = int(s["end_row"])
        mask = (rows >= start) & (rows < end)

        if not mask.any():
            continue

        local = rows[mask] - start
        arr = np.load(SHARD_ROOT / s["file"], mmap_mode="r")
        out[mask] = np.asarray(arr[local], dtype=np.float32)
        filled[mask] = True

    if not filled.all():
        raise RuntimeError(f"Failed to load {np.sum(~filled)} rows")

    return out

rng = np.random.default_rng(SEED)
audit_rows = np.sort(rng.choice(N_DOCS, size=5000, replace=False))
audit = load_rows_from_shards(audit_rows)
norms = np.linalg.norm(audit, axis=1)

print("finite:", np.isfinite(audit).all())
print("nonzero:", np.count_nonzero(audit))
print("norm min/mean/max:", norms.min(), norms.mean(), norms.max())

assert np.isfinite(audit).all()
assert np.count_nonzero(audit) > 0
assert norms.min() > 0.995
assert norms.max() < 1.005

print("VECTOR AUDIT — PASS")


finite: True
nonzero: 1920000
norm min/mean/max: 0.99987763 1.0000004 1.0001249
VECTOR AUDIT — PASS


## 2. Reload DEV queries and qrels


In [ ]:
queries = np.load(QUERY_EMB, mmap_mode="r")

with open(QUERY_IDS, "r", encoding="utf-8") as f:
    query_ids = [x.strip() for x in f if x.strip()]

with open(SPLIT_MANIFEST, "r", encoding="utf-8") as f:
    split = json.load(f)

dev_ids = [str(x) for x in split["dev_query_ids"]]

assert queries.shape == (97852, DIM)
assert len(dev_ids) == 5447
assert split["test_qrels_relevance_values_accessed"] is False
assert split["test_retrieval_performed"] is False
assert split["test_outcomes_observed"] is False

query_row = {qid:i for i,qid in enumerate(query_ids)}
confirmation_rows = np.asarray([query_row[q] for q in dev_ids], dtype=np.int64)
Q_DEV = np.ascontiguousarray(np.asarray(queries[confirmation_rows], dtype=np.float32))

qnorm = np.linalg.norm(Q_DEV, axis=1)
assert qnorm.min() > 0.99
assert qnorm.max() < 1.01

dev = pd.read_csv(DEV_QRELS, sep="\t")
dev["query-id"] = dev["query-id"].astype(str)
dev["corpus-id"] = dev["corpus-id"].astype(str)
assert set(dev["query-id"]) == set(dev_ids)

unique_doc_ids = dev["corpus-id"].drop_duplicates().tolist()

with sqlite3.connect(str(CORPUS_DB)) as con:
    con.execute("CREATE TEMP TABLE requested_ids (doc_id TEXT PRIMARY KEY)")
    con.executemany(
        "INSERT INTO requested_ids(doc_id) VALUES (?)",
        [(x,) for x in unique_doc_ids],
    )
    mapped = pd.read_sql_query(
        """
        SELECT r.doc_id, d.row_id
        FROM requested_ids r
        LEFT JOIN documents d ON d.doc_id = r.doc_id
        """,
        con,
    )

assert mapped["row_id"].notna().all()
mapped["row_id"] = mapped["row_id"].astype(np.int64)

dev = dev.merge(
    mapped,
    left_on="corpus-id",
    right_on="doc_id",
    how="left",
    validate="many_to_one",
).rename(columns={"row_id":"corpus-row"})

dev["corpus-row"] = dev["corpus-row"].astype(np.int64)

hotpot_dev_qrels = {}
for qid, g in dev.groupby("query-id"):
    hotpot_dev_qrels[str(qid)] = set(
        g.loc[g["score"] > 0, "corpus-row"].astype(np.int64).tolist()
    )

assert len(hotpot_dev_qrels) == 5447
assert set(map(len, hotpot_dev_qrels.values())) == {2}

print("DEV ALIGNMENT — PASS")


DEV ALIGNMENT — PASS


## 3. Representation validity gate


In [ ]:
positive_query_rows = np.asarray(
    [query_row[str(q)] for q in dev["query-id"]],
    dtype=np.int64,
)
positive_doc_rows = dev["corpus-row"].to_numpy(np.int64)

Q_POS = np.asarray(queries[positive_query_rows], dtype=np.float32)
D_POS = load_rows_from_shards(positive_doc_rows)
positive_scores = np.sum(Q_POS * D_POS, axis=1)

rng = np.random.default_rng(SEED)
random_rows = rng.integers(0, N_DOCS, size=len(positive_doc_rows), dtype=np.int64)
D_RAND = load_rows_from_shards(random_rows)
random_scores = np.sum(Q_POS * D_RAND, axis=1)

margin = positive_scores - random_scores

print("positive mean:", float(positive_scores.mean()))
print("random mean:", float(random_scores.mean()))
print("mean margin:", float(margin.mean()))
print("fraction positive:", float(np.mean(margin > 0)))

assert positive_scores.mean() > random_scores.mean()
assert np.mean(margin > 0) > 0.95

print("REPRESENTATION GATE — PASS")


positive mean: 0.7168717980384827
random mean: 0.3556794822216034
mean margin: 0.36119234561920166
fraction positive: 0.9995410317606022
REPRESENTATION GATE — PASS


## 4. Deterministic 200k training sample


In [ ]:
rng = np.random.default_rng(SEED)
train_rows = np.sort(rng.choice(N_DOCS, size=TRAIN_DOCS, replace=False)).astype(np.int64)
np.save(OUT / "training_rows.int64.npy", train_rows, allow_pickle=False)

t0 = time.time()
train_x = load_rows_from_shards(train_rows)
train_norms = np.linalg.norm(train_x, axis=1)

print("train:", train_x.shape)
print("norm min/mean/max:", train_norms.min(), train_norms.mean(), train_norms.max())
print("elapsed:", round(time.time()-t0, 1), "s")

assert train_x.shape == (TRAIN_DOCS, DIM)
assert train_norms.min() > 0.995
assert train_norms.max() < 1.005

print("TRAINING SAMPLE — PASS")


train: (200000, 384)
norm min/mean/max: 0.9998581 0.99999994 1.0001374
elapsed: 7.6 s
TRAINING SAMPLE — PASS


## 5. Shard-streamed FAISS builders


In [ ]:
PROCESS = psutil.Process(os.getpid())

def atomic_write_index(index, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    if tmp.exists():
        tmp.unlink()
    faiss.write_index(index, str(tmp))
    os.replace(tmp, path)

def stream_add(index, label):
    added = 0
    t0 = time.time()

    for i, s in enumerate(shards):
        arr = np.load(SHARD_ROOT / s["file"], mmap_mode="r")
        xb = np.ascontiguousarray(np.asarray(arr, dtype=np.float32))
        index.add(xb)
        added += len(xb)
        del xb, arr

        if (i+1) % 10 == 0 or i+1 == len(shards):
            print(
                f"{label}: {added:,}/{N_DOCS:,} "
                f"({100*added/N_DOCS:.2f}%) "
                f"elapsed={(time.time()-t0)/60:.1f} min"
            )
            gc.collect()

    assert added == N_DOCS
    return index

def build_index(kind, path):
    path = Path(path)

    if path.is_file():
        print("CACHE EXISTS:", path)
        index = faiss.read_index(str(path))
        index.nprobe = NPROBE
        return index

    quantizer = faiss.IndexFlatIP(DIM)

    if kind == "pq32":
        index = faiss.IndexIVFPQ(
            quantizer, DIM, NLIST, 32, NBITS, faiss.METRIC_INNER_PRODUCT
        )
    elif kind == "pq64":
        index = faiss.IndexIVFPQ(
            quantizer, DIM, NLIST, 64, NBITS, faiss.METRIC_INNER_PRODUCT
        )
    elif kind == "sq8":
        index = faiss.IndexIVFScalarQuantizer(
            quantizer,
            DIM,
            NLIST,
            faiss.ScalarQuantizer.QT_8bit,
            faiss.METRIC_INNER_PRODUCT,
        )
    else:
        raise ValueError(kind)

    print("TRAIN:", kind)
    index.train(np.ascontiguousarray(train_x, dtype=np.float32))
    assert index.is_trained

    stream_add(index, kind)

    assert index.ntotal == N_DOCS
    index.nprobe = NPROBE

    atomic_write_index(index, path)
    print(kind, "SHA-256:", sha256_file(path))
    print(kind, "BUILD — COMPLETE")
    return index


## 6. Build PQ32 → PQ64 → SQ8


In [ ]:
pq32 = build_index("pq32", PQ32_PATH)
assert pq32.ntotal == N_DOCS
del pq32
gc.collect()

pq64 = build_index("pq64", PQ64_PATH)
assert pq64.ntotal == N_DOCS
del pq64
gc.collect()

sq8 = build_index("sq8", SQ8_PATH)
assert sq8.ntotal == N_DOCS
del sq8
gc.collect()

print("ALL THREE INDEXES — BUILT/CACHED")


TRAIN: pq32
pq32: 200,000/5,233,329 (3.82%) elapsed=0.1 min
pq32: 400,000/5,233,329 (7.64%) elapsed=0.3 min
pq32: 600,000/5,233,329 (11.46%) elapsed=0.4 min
pq32: 800,000/5,233,329 (15.29%) elapsed=0.5 min
pq32: 1,000,000/5,233,329 (19.11%) elapsed=0.7 min
pq32: 1,200,000/5,233,329 (22.93%) elapsed=0.8 min
pq32: 1,400,000/5,233,329 (26.75%) elapsed=1.0 min
pq32: 1,600,000/5,233,329 (30.57%) elapsed=1.1 min
pq32: 1,800,000/5,233,329 (34.39%) elapsed=1.2 min
pq32: 2,000,000/5,233,329 (38.22%) elapsed=1.4 min
pq32: 2,200,000/5,233,329 (42.04%) elapsed=1.5 min
pq32: 2,400,000/5,233,329 (45.86%) elapsed=1.6 min
pq32: 2,600,000/5,233,329 (49.68%) elapsed=1.8 min
pq32: 2,800,000/5,233,329 (53.50%) elapsed=1.9 min
pq32: 3,000,000/5,233,329 (57.32%) elapsed=2.0 min
pq32: 3,200,000/5,233,329 (61.15%) elapsed=2.2 min
pq32: 3,400,000/5,233,329 (64.97%) elapsed=2.3 min
pq32: 3,600,000/5,233,329 (68.79%) elapsed=2.4 min
pq32: 3,800,000/5,233,329 (72.61%) elapsed=2.6 min
pq32: 4,000,000/5,233,329 (76

## 7. IVF population audit


In [ ]:
def population_audit(path, label):
    index = faiss.read_index(str(path))
    index.nprobe = NPROBE
    ivf = faiss.extract_index_ivf(index)

    sizes = np.asarray(
        [ivf.invlists.list_size(i) for i in range(ivf.nlist)],
        dtype=np.int64,
    )

    summary = {
        "condition": label,
        "sha256": sha256_file(path),
        "bytes": int(path.stat().st_size),
        "ntotal": int(index.ntotal),
        "empty_lists": int(np.sum(sizes == 0)),
        "median_list_size": float(np.median(sizes)),
        "max_list_size": int(sizes.max()),
        "sum_list_sizes": int(sizes.sum()),
    }

    print(summary)

    assert sizes.sum() == N_DOCS
    assert np.sum(sizes > 0) > int(0.90 * NLIST)
    assert sizes.max() < int(0.02 * N_DOCS)

    return summary

index_audits = [
    population_audit(PQ32_PATH, "PQ32"),
    population_audit(PQ64_PATH, "PQ64"),
    population_audit(SQ8_PATH, "SQ8"),
]

print("INDEX POPULATION AUDITS — PASS")


{'condition': 'PQ32', 'sha256': 'dcd99cd8e91fce24e561a1d0693dd5f5ed7725facd364b12478bada9ad721fe8', 'bytes': 216050780, 'ntotal': 5233329, 'empty_lists': 0, 'median_list_size': 1135.5, 'max_list_size': 6709, 'sum_list_sizes': 5233329}
{'condition': 'PQ64', 'sha256': 'de5dca03390eb91d8814c3a370d588906a65403d7e9b1ca03f745531df30c069', 'bytes': 383517308, 'ntotal': 5233329, 'empty_lists': 0, 'median_list_size': 1135.5, 'max_list_size': 6709, 'sum_list_sizes': 5233329}
{'condition': 'SQ8', 'sha256': '3cca7dc6e3c56c1dc9aee7a73d158e0f2e6aeb0157d5af515cd583fd19937587', 'bytes': 2057792448, 'ntotal': 5233329, 'empty_lists': 0, 'median_list_size': 1135.5, 'max_list_size': 6709, 'sum_list_sizes': 5233329}
INDEX POPULATION AUDITS — PASS


## 8. Retrieval metrics


In [ ]:
def evaluate_rankings(qids, ranked_ids, k=10):
    recall = np.empty(len(qids), np.float32)
    mrr = np.empty(len(qids), np.float32)
    ndcg = np.empty(len(qids), np.float32)

    discounts = 1.0 / np.log2(np.arange(2, k+2))

    for i, qid in enumerate(qids):
        rel = hotpot_dev_qrels[str(qid)]
        ranked = ranked_ids[i, :k]

        hits = np.asarray(
            [1.0 if int(d) in rel else 0.0 for d in ranked],
            dtype=np.float32,
        )

        recall[i] = hits.sum() / len(rel)
        pos = np.flatnonzero(hits)
        mrr[i] = 1.0 / (int(pos[0])+1) if len(pos) else 0.0

        dcg = float((hits * discounts).sum())
        idcg = float(discounts[:min(len(rel), k)].sum())
        ndcg[i] = dcg / idcg if idcg > 0 else 0.0

    return recall, mrr, ndcg


## 9. 100-query smoke test


In [ ]:
SMOKE_N = 100
smoke_qids = dev_ids[:SMOKE_N]
Q_SMOKE = np.ascontiguousarray(Q_DEV[:SMOKE_N], dtype=np.float32)

smoke_rows = []

for label, path in [
    ("PQ32", PQ32_PATH),
    ("PQ64", PQ64_PATH),
    ("SQ8", SQ8_PATH),
]:
    index = faiss.read_index(str(path))
    index.nprobe = NPROBE

    _, ids = index.search(Q_SMOKE, SEARCH_K)
    r, m, n = evaluate_rankings(smoke_qids, ids, TOP_K)

    print(label, "R@10=", float(r.mean()), "MRR@10=", float(m.mean()), "nDCG@10=", float(n.mean()))
    print(label, "first query:", ids[0, :10])

    assert float(r.mean()) > 0.0

    smoke_rows.append({
        "condition": label,
        "recall@10": float(r.mean()),
        "mrr@10": float(m.mean()),
        "ndcg@10": float(n.mean()),
    })

    del index
    gc.collect()

smoke_df = pd.DataFrame(smoke_rows)
display(smoke_df)

print("100-QUERY SMOKE — PASS")


PQ32 R@10= 0.5450000166893005 MRR@10= 0.6015475988388062 nDCG@10= 0.491709440946579
PQ32 first query: [2157697 3750881 1359089 3338867  504945 4101851 2131931  279227  698749
 4013074]
PQ64 R@10= 0.6650000214576721 MRR@10= 0.7107064127922058 nDCG@10= 0.60922771692276
PQ64 first query: [2131931  919754    2869 3591362 1369067 2166848 2735829 1499472  698749
  575243]
SQ8 R@10= 0.7149999737739563 MRR@10= 0.7611904740333557 nDCG@10= 0.6588152050971985
SQ8 first query: [2131931 2735829  397906  919754 2157697 4705494  949390 4626123 1369067
  295150]


,condition,recall@10,mrr@10,ndcg@10
0,PQ32,0.545,0.601548,0.491709
1,PQ64,0.665,0.710706,0.609228
2,SQ8,0.715,0.761190,0.658815


100-QUERY SMOKE — PASS


## 10. Full 5,447-query DEV baseline


In [ ]:
baseline_rows = []
per_query_frames = []

for label, path in [
    ("PQ32", PQ32_PATH),
    ("PQ64", PQ64_PATH),
    ("SQ8", SQ8_PATH),
]:
    print("FULL DEV:", label)

    index = faiss.read_index(str(path))
    index.nprobe = NPROBE

    t0 = time.time()
    _, ids = index.search(Q_DEV, SEARCH_K)
    elapsed = time.time() - t0

    r, m, n = evaluate_rankings(dev_ids, ids, TOP_K)

    baseline_rows.append({
        "condition": label,
        "queries": len(dev_ids),
        "recall@10": float(r.mean()),
        "mrr@10": float(m.mean()),
        "ndcg@10": float(n.mean()),
        "search_seconds": float(elapsed),
        "qps": float(len(dev_ids)/elapsed),
    })

    per_query_frames.append(
        pd.DataFrame({
            "query_id": dev_ids,
            "condition": label,
            "recall@10": r,
            "mrr@10": m,
            "ndcg@10": n,
        })
    )

    print(baseline_rows[-1])

    del index
    gc.collect()

baseline_df = pd.DataFrame(baseline_rows)
per_query_df = pd.concat(per_query_frames, ignore_index=True)

display(baseline_df)

assert (baseline_df["recall@10"] > 0).all()
assert (baseline_df["ndcg@10"] > 0).all()

pq32_r = float(baseline_df.loc[baseline_df["condition"]=="PQ32", "recall@10"].iloc[0])
sq8_r  = float(baseline_df.loc[baseline_df["condition"]=="SQ8", "recall@10"].iloc[0])

assert sq8_r >= 0.80 * pq32_r

print("FULL DEV BASELINE — PASS")


FULL DEV: PQ32
{'condition': 'PQ32', 'queries': 5447, 'recall@10': 0.5350651741027832, 'mrr@10': 0.6024250984191895, 'ndcg@10': 0.48980867862701416, 'search_seconds': 5.809344053268433, 'qps': 937.6273723942082}
FULL DEV: PQ64
{'condition': 'PQ64', 'queries': 5447, 'recall@10': 0.6421883702278137, 'mrr@10': 0.7406182885169983, 'ndcg@10': 0.6112769246101379, 'search_seconds': 9.72110390663147, 'qps': 560.3273097702624}
FULL DEV: SQ8
{'condition': 'SQ8', 'queries': 5447, 'recall@10': 0.6827611327171326, 'mrr@10': 0.8006231188774109, 'ndcg@10': 0.6632661819458008, 'search_seconds': 7.216834306716919, 'qps': 754.763067641766}


,condition,queries,recall@10,mrr@10,ndcg@10,search_seconds,qps
0,PQ32,5447,0.535065,0.602425,0.489809,5.809344,937.627372
1,PQ64,5447,0.642188,0.740618,0.611277,9.721104,560.327310
2,SQ8,5447,0.682761,0.800623,0.663266,7.216834,754.763068


FULL DEV BASELINE — PASS


## 11. Save evidence


In [ ]:
smoke_df.to_csv(OUT / "smoke_100q.csv", index=False)
baseline_df.to_csv(OUT / "hotpotqa_dev_baseline.csv", index=False)
per_query_df.to_csv(OUT / "hotpotqa_dev_baseline_per_query.csv", index=False)
pd.DataFrame(index_audits).to_csv(OUT / "index_population_audit.csv", index=False)

report = {
    "status": "HOTPOTQA_FIDELITY_INDEX_REBUILD_V07_COMPLETE",
    "dataset": "HotpotQA",
    "corpus_rows": N_DOCS,
    "dev_queries": len(dev_ids),
    "nlist": NLIST,
    "nprobe": NPROBE,
    "training_documents": TRAIN_DOCS,
    "source_shard_manifest_sha256": sha256_file(SHARD_MANIFEST),
    "indexes": {
        "PQ32": {"path": str(PQ32_PATH), "sha256": sha256_file(PQ32_PATH)},
        "PQ64": {"path": str(PQ64_PATH), "sha256": sha256_file(PQ64_PATH)},
        "SQ8": {"path": str(SQ8_PATH), "sha256": sha256_file(SQ8_PATH)},
    },
    "smoke_100q": smoke_df.to_dict(orient="records"),
    "full_dev_baseline": baseline_df.to_dict(orient="records"),
    "index_population_audit": index_audits,
    "test_qrels_accessed": False,
    "test_retrieval_performed": False,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
}

REPORT = OUT / "report.json"
REPORT.write_text(
    json.dumps(report, indent=2, ensure_ascii=False, default=float),
    encoding="utf-8",
)

sha = sha256_file(REPORT)
(OUT / "REPORT_SHA256.txt").write_text(sha + "\n", encoding="utf-8")

print("Saved:", OUT)
print("Report SHA-256:", sha)


Saved: /content/drive/MyDrive/rag-pq-checkpoints/arc-v0/hotpotqa-fidelity-index-rebuild-v07/20260816-101236
Report SHA-256: a4e46d059721ec05b15e386deba5e8ed72b968712d03a01570aa6c934a2fc703


## Gate for ARC-v0.8

Proceed to the sealed HotpotQA H1–H4 mechanism replication only if:

- corpus shard audits pass;
- representation gate passes;
- all three IVF population audits pass;
- all 100-query smoke metrics are non-zero;
- all full-DEV baseline metrics are non-zero;
- SQ8 does not catastrophically underperform PQ32;
- TEST remains untouched.
